In [4]:
import os
import json
import requests
import pandas as pd
import numpy as np

In [5]:
df = pd.read_csv("../data/raw/election_by_county.csv")

df

,county_code,election_year,state,eligible_voters,number_voters,valid_votes,invalid_votes,turnout,cdu,csu,...,zentrum,election_date,area,population,flag_unsuccessful_naive_merge,total_votes,cdu_csu,far_right,far_left,far_left_w_linke
0,1001,1990,1,69563,50485,50036,449,0.725745,0.370533,0.0,...,0.0,1990-12-02,56.36,86600,NaN,50036,0.370533,0.014330,0.000000,0.004157
1,1001,1994,1,68987,52379,51823,556,0.759259,0.344191,0.0,...,0.0,1994-10-16,56.44,87900,NaN,51823,0.344191,0.009629,0.000289,0.013797
2,1001,1998,1,65755,50560,49862,698,0.768915,0.291565,0.0,...,0.0,1998-09-27,56.44,84700,NaN,49862,0.291565,0.020697,0.000000,0.018290
3,1001,2002,1,65740,49054,48553,501,0.746182,0.295800,0.0,...,0.0,2002-09-22,56.38,84700,NaN,48553,0.295800,0.004284,0.000000,0.017692
4,1001,2005,1,66970,49002,48235,767,0.731701,0.288380,0.0,...,0.0,2005-09-18,56.38,86100,NaN,48235,0.288380,0.008852,0.000705,0.064269
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3995,16074,2025,16,65981,55149,54806,343,0.835832,0.196183,0.0,...,NaN,2025-02-23,815.24,82513,0.0,54806,0.196183,0.386764,0.000839,0.144273
3996,16075,2025,16,63125,51544,51203,341,0.816539,0.174951,0.0,...,NaN,2025-02-23,1151.30,79030,0.0,51203,0.174951,0.460598,0.000957,0.128137
3997,16076,2025,16,77827,63523,63045,478,0.816208,0.189547,0.0,...,NaN,2025-02-23,845.98,96102,0.0,63045,0.189547,0.437148,0.000952,0.122532
3998,16077,2025,16,70997,55631,55172,459,0.783568,0.173675,0.0,...,NaN,2025-02-23,569.39,87807,0.0,55172,0.173675,0.457623,0.001160,0.119481


In [6]:
def afd_in_election(df, year):
    df_year = df[df['election_year'] == year].copy()
    df_year['afd_votes'] = (df_year['afd'] / 100) * df_year['valid_votes']
    state_afd = df_year.groupby('state').agg({
        'afd_votes': 'sum',
        'valid_votes': 'sum'
    }).reset_index()
    state_afd['afd_share_pct'] = (state_afd['afd_votes'] / state_afd['valid_votes']) * 100
    
    return state_afd
    
   

In [7]:
afd_2017 = afd_in_election(df, 2017)
afd_2021 = afd_in_election(df, 2021)
afd_2025 = afd_in_election(df, 2025)

In [8]:
afd_2017['year'] = 2017
afd_2021['year'] = 2021
afd_2025['year'] = 2025

In [9]:
afd_panel = pd.concat([afd_2017, afd_2021, afd_2025], ignore_index=True)

In [10]:
afd_panel

,state,afd_votes,valid_votes,afd_share_pct,year
0,1,1403.620000,1715641,0.081813,2017
1,2,765.110000,978118,0.078223,2017
2,3,4223.620000,4646976,0.090890,2017
3,4,332.440000,332323,0.100035,2017
4,5,9284.250000,9853377,0.094224,2017
5,6,3987.981654,3348900,0.119083,2017
6,7,2656.880000,2362506,0.112460,2017
7,8,7304.990000,5992968,0.121893,2017
8,9,9164.581794,7393210,0.123959,2017
9,10,589.200000,585258,0.100674,2017


In [11]:
state_mapping = {
    1: 'Schleswig-Holstein',
    2: 'Hamburg',
    3: 'Niedersachsen',
    4: 'Bremen',
    5: 'Nordrhein-Westfalen',
    6: 'Hessen',
    7: 'Rheinland-Pfalz',
    8: 'Baden-Württemberg',
    9: 'Bayern',
    10: 'Saarland',
    11: 'Berlin',
    12: 'Brandenburg',
    13: 'Mecklenburg-Vorpommern',
    14: 'Sachsen',
    15: 'Sachsen-Anhalt',
    16: 'Thüringen'
}

In [12]:
afd_panel['state_name'] = afd_panel['state'].map(state_mapping)

In [13]:
afd_panel

,state,afd_votes,valid_votes,afd_share_pct,year,state_name
0,1,1403.620000,1715641,0.081813,2017,Schleswig-Holstein
1,2,765.110000,978118,0.078223,2017,Hamburg
2,3,4223.620000,4646976,0.090890,2017,Niedersachsen
3,4,332.440000,332323,0.100035,2017,Bremen
4,5,9284.250000,9853377,0.094224,2017,Nordrhein-Westfalen
5,6,3987.981654,3348900,0.119083,2017,Hessen
6,7,2656.880000,2362506,0.112460,2017,Rheinland-Pfalz
7,8,7304.990000,5992968,0.121893,2017,Baden-Württemberg
8,9,9164.581794,7393210,0.123959,2017,Bayern
9,10,589.200000,585258,0.100674,2017,Saarland


In [17]:
afd_panel.to_csv('../data/processed/afd_election_by_state.csv', index=False)
